# Go Fine-tuning Pipeline — Google Colab

Train a Go/Golang coding model on Colab GPU.

**Setup:** Runtime → Change runtime type → Pick GPU + Enable **High-RAM**

| GPU | VRAM | Recommended |
|-----|------|-------------|
| A100 | 40GB | Best (Pro only) |
| L4 | 24GB | Great |
| T4 | 16GB | OK (free tier) |

## 1. Setup

In [ ]:
# Clone repo
!git clone https://github.com/slotheth/prototype-model.git
%cd prototype-model

# Mount Google Drive for persistent storage (survives session crashes)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/go-coder-training

In [ ]:
# Install dependencies (Colab already has PyTorch + CUDA)
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q peft trl datasets accelerate bitsandbytes
!pip install -q python-dotenv sentencepiece protobuf openai

# Required for Qwen3.5 (GatedDeltaNet architecture)
!pip install -q flash-linear-attention causal-conv1d

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Configure

Edit the cell below with your settings.
A100 40GB can handle higher seq_length and LoRA rank.

In [ ]:
# Write .env config — EDIT YOUR HF_TOKEN HERE
env_content = """
HF_TOKEN=your_huggingface_token_here

MODEL_NAME=Jackrong/Qwen3.5-4B-Claude-4.6-Opus-Reasoning-Distilled

# A100 40GB optimized settings
MAX_SEQ_LENGTH=2048
LORA_R=32
LORA_ALPHA=32
NUM_TRAIN_EPOCHS=3
LEARNING_RATE=2e-4
GRADIENT_ACCUMULATION_STEPS=8
PER_DEVICE_TRAIN_BATCH_SIZE=2
SAVE_STEPS=200

# Save to Google Drive (survives session crashes)
OUTPUT_DIR=/content/drive/MyDrive/go-coder-training/outputs
""".strip()

with open('.env', 'w') as f:
    f.write(env_content)

print('Config written to .env')
print('Output dir: Google Drive (persistent)')
print()
print('>>> EDIT THE CELL ABOVE: set your HF_TOKEN before running!')

In [ ]:
# Verify config loads correctly
import config as cfg
print(f'Model:      {cfg.MODEL_NAME}')
print(f'Seq length: {cfg.MAX_SEQ_LENGTH}')
print(f'LoRA R:     {cfg.LORA_R}')
print(f'Epochs:     {cfg.NUM_TRAIN_EPOCHS}')
print(f'Batch size: {cfg.PER_DEVICE_TRAIN_BATCH_SIZE} x {cfg.GRADIENT_ACCUMULATION_STEPS} = {cfg.PER_DEVICE_TRAIN_BATCH_SIZE * cfg.GRADIENT_ACCUMULATION_STEPS}')

## 3. Prepare Dataset

In [ ]:
!python prepare_data.py

## 4. Train

In [ ]:
# Train (auto-resumes from latest checkpoint if session crashed)
import glob
checkpoints = sorted(glob.glob('/content/drive/MyDrive/go-coder-training/outputs/checkpoint-*'))
if checkpoints:
    print(f'Found checkpoint: {checkpoints[-1]}')
    print('Resuming training...')
    !python train.py --resume {checkpoints[-1]}
else:
    print('Starting fresh training...')
    !python train.py

## 5. Export to GGUF

In [ ]:
# Setup llama.cpp for GGUF conversion
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q gguf

In [ ]:
# Export as q8_0 (good quality, ~4GB for 3B model)
!python export.py --quantize q8_0

## 6. Download Results

Download the trained model to your local machine.

In [ ]:
# Option A: Download GGUF directly
from google.colab import files
files.download('outputs/go-coder.gguf')

In [ ]:
# Option B: Push LoRA adapters to HuggingFace Hub
from huggingface_hub import HfApi
import config as cfg

api = HfApi(token=cfg.HF_TOKEN)
api.upload_folder(
    folder_path=str(cfg.LORA_DIR),
    repo_id='your-username/go-coder-lora',  # CHANGE THIS
    repo_type='model',
    create_pr=False,
)

In [ ]:
# Option C: Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

!cp outputs/go-coder.gguf /content/drive/MyDrive/
!cp outputs/merged-model/Modelfile /content/drive/MyDrive/
print('Saved to Google Drive!')

## 7. Quick Test (Optional)

In [ ]:
!python infer.py "Write a Go HTTP server with graceful shutdown"